Load Necessary Packages

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.stem import SnowballStemmer

from sentence_transformers import SentenceTransformer

import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, precision_score, accuracy_score

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

import re, warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier

SEED = 42
np.random.seed(SEED)

In [ ]:
# Uncomment and run once:
# !pip install bertopic datasets sentence-transformers umap-learn hdbscan plotly

# Verify installation
import bertopic
print(f"✅ BERTopic version: {bertopic.__version__}")

In [ ]:
data = pd.read_csv("data.csv")
data

In [ ]:
title = data['title'].tolist()
topic = data['topic'].tolist()

In [ ]:
data['topic'].unique()

In [ ]:
print(f"Sampled dataset size: {len(data):,} documents")
print("\nTopic distribution:")
print(data["topic"].value_counts())

EDA

In [ ]:
# ── Document length distribution ──────────────────────────────────────────
data["word_count"] = data["title"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Overall histogram
axes[0].hist(data["word_count"], bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("Document Length Distribution", fontsize=13)
axes[0].set_xlabel("Word Count")
axes[0].set_ylabel("Frequency")
axes[0].axvline(data["word_count"].median(), color="tomato",
                linestyle="--", label=f'Median: {data["word_count"].median():.0f}')
axes[0].legend()

# By category
colors = ["steelblue", "darkorange", "seagreen", "tomato"]
for i, (cat, grp) in enumerate(data.groupby("topic")):
    axes[1].hist(grp["word_count"], bins=30, alpha=0.5, label=cat, color=colors[i])
axes[1].set_title("Document Length by Category", fontsize=13)
axes[1].set_xlabel("Word Count")
axes[1].legend()

plt.tight_layout()
plt.show()

print(data[["word_count"]].describe().round(1))

Remove Stop Words

In [ ]:
# !pip install scikit-learn nltk gensim numpy pandas

# Try to load NLTK stopwords; fall back to a small built-in list if download fails
try:
    nltk.download('stopwords', quiet=True)
    from nltk.corpus import stopwords
    STOP_WORDS = set(stopwords.words('english'))
except Exception:
    STOP_WORDS = {'a','an','the','and','or','but','if','of','at','by','for','with',
                  'about','to','from','in','on','is','are','was','were','be','been',
                  
                  'being','have','has','had','do','does','did','this','that','these',
                  'those','i','you','he','she','it','we','they','them','their','our',
                  'my','your','his','her','its'}

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

stop_words = STOP_WORDS
print(f'Using {len(stop_words)} English stop words.')

# remove non-english words that appear in data
# included also due to frequency
extra_words = {'also', 'περὶ', 'σπέρμα', 'τηθύς',
               'τραχεῖα', 'φυτά', 'φυτόν',
                'φυτῶν', 'המלח', 'ים', 'מעבדות', 'פרמייר', 
                'ἀγγεῖον', 'ἀρτηρία', 'ストーンオーシャン',
                '達爾膚生醫科技', '대우조선해양', '빅오션', '케이뷰티','한화오션',
                "list", "lists", "smith", "hope", "Lists", "of", "List"
}

stop_words.update(extra_words)
print(f'Using {len(stop_words)} English stop words.')

stop_words.remove('σπέρμα')
print(f'Using {len(stop_words)} English stop words.')

In [ ]:
stop_words

In [ ]:
tokenizer = CountVectorizer().build_tokenizer()
stemmer = SnowballStemmer('english')

Pre-Process for TF-IDF

In [ ]:
def preprocess(text):
    tokens = tokenizer(text)  # tokenize
    tokens = [stemmer.stem(t) for t in tokens] # stem
    tokens = [t for t in tokens if t not in stop_words]  # drop stop words
    return tokens

processed = [preprocess(d) for d in title]
for i, p in enumerate(processed, 1):
    print(f'Article{i}: {p}')

In [ ]:
# pass preprocess function as the tokenizer
bow_vectorizer = CountVectorizer(tokenizer=preprocess, token_pattern=None, lowercase=False)
X_bow = bow_vectorizer.fit_transform(title)
vocab = bow_vectorizer.get_feature_names_out()

print(f'Vocabulary size: {len(vocab)}')
print(f'Vocabulary: {list(vocab)}')
print(f'Matrix shape: {X_bow.shape}  (199 Articles x {len(vocab)} words)')

In [ ]:
# display the document-feature matrix as a nice table
df_bow = pd.DataFrame(
    X_bow.toarray(),
    columns=vocab,
    index=[f'Article{i}' for i in range(1, 200)]
)
df_bow

In [ ]:
# cosine similarity on BOW vectors
sim_bow = cosine_similarity(X_bow)

df_sim_bow = pd.DataFrame(
    sim_bow.round(3),
    columns=[f'Article{i}' for i in range(1, 200)],
    index=[f'Article{i}' for i in range(1, 200)]
)
df_sim_bow

Cleaning for Embedding Data

In [ ]:
def clean_text(text: str) -> str:
    """Light cleaning for news text."""
    # Decode HTML entities
    text = text.replace("&lt;", "<").replace("&gt;", ">").replace("&amp;", "&")
    text = text.replace("&quot;", '"').replace("&#39;", "'")
    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", "", text)
    # Collapse multiple spaces / newlines
    text = re.sub(r"\s+", " ", text).strip()
    # Remove Punctuation
    # text = re.sub(r"[^\w\s]", "", text)
    return text

data["clean_text"] = data["title"].apply(clean_text)

# Drop very short documents
before = len(data)
data = data[data["clean_text"].str.split().str.len() >= 1].reset_index(drop=True)
print(f"Dropped {before - len(data)} too-short documents. Remaining: {len(data):,}")

# The list of strings we'll feed to BERTopic
docs = data["clean_text"].tolist()
print(f"\n✅ Ready: {len(docs):,} documents")

print(docs[:5])

TF-IDF

In [ ]:
tfidf_vectorizer = TfidfVectorizer(tokenizer=preprocess, token_pattern=None, lowercase=False)
X_tfidf = tfidf_vectorizer.fit_transform(title) # used for randomforest

df_tfidf = pd.DataFrame(
    X_tfidf.toarray().round(3),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f'Article {i}' for i in range(1, 200)]
)
df_tfidf

Embedding

In [ ]:
# method b
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
print("Generating embeddings...")
embeddings = embedding_model.encode(docs, show_progress_bar=True, batch_size=64)

print(f"\n✅ Embeddings shape: {embeddings.shape}")
print(f"   → {embeddings.shape[0]} documents × {embeddings.shape[1]} dimensions")

Split Data

In [ ]:
X_tf = X_tfidf
y = topic

X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(
    X_tf, y, test_size=0.2, random_state=SEED, stratify=y
)

In [ ]:
X_emb = embeddings
y = topic

X_train_emd, X_test_emd, y_train_emd, y_test_emb = train_test_split(
    X_emb, y, test_size=0.2, random_state=SEED, stratify=y
)

Random Forest Classifer + K-Fold Validation

In [ ]:
k = 5
# trying different parameters
n_estm = [100, 200, 300, 400, 500]
max_dpth = [2, 3, 5, 7, 9, 10, 11]

randomforest = RandomForestClassifier(random_state=SEED)

# cross val + hyperparameter tuning
models_tfidf = GridSearchCV(
    randomforest,
    param_grid=dict(n_estimators=n_estm, max_depth=max_dpth),
    cv=k,
    scoring='f1_macro',
    return_train_score=True
)

# cross val + hyperparameter tuning
models_emb = GridSearchCV(
    randomforest,
    param_grid=dict(n_estimators=n_estm, max_depth=max_dpth),
    cv=k,
    scoring='f1_macro',
    return_train_score=True
)

models_tfidf.fit(X_train_tfidf, y_train_tfidf)
models_emb.fit(X_train_emd, y_train_emd)

Find the Best Parameters

In [ ]:
# best parameters for tf-idf
best_model_tfidf = models_tfidf.best_estimator_
print(models_tfidf.best_params_)

# best parameters for embedding
best_model_emb = models_emb.best_estimator_
print(models_emb.best_params_)

TF-IDF Feature Importance

In [ ]:
# extract important features
feature_names = tfidf_vectorizer.get_feature_names_out()
importance = best_model_tfidf.feature_importances_

feat_df = pd.DataFrame({
    'word': feature_names,
    'importance': importance
})

# top 10 features
top = feat_df.sort_values(
    by='importance',
    ascending=False
)

top10 = top.head(10)

# create bar char
plt.bar(top10["word"], top10["importance"])
plt.ylabel("Importance")
plt.xlabel("Word")
plt.title("Top 10 Important Feature in TF-IDF RandomForest Model")
plt.show()

TF-IDF Recall, Precision, F1

In [ ]:
y_pred = best_model_tfidf.predict(X_test_tfidf)

# evaluate the model
print(classification_report(y_test_tfidf, y_pred))

Embedding Recall, Precision, F1

In [ ]:
y_pred = best_model_emb.predict(X_test_emd)

# evaluate the model
print(classification_report(y_test_emb, y_pred))

Feature Selection

In [ ]:
# extract important features
feature_names = tfidf_vectorizer.get_feature_names_out()
importance = best_model_tfidf.feature_importances_

feat_df = pd.DataFrame({
    'word': feature_names,
    'importance': importance
})

# top features
top = feat_df.sort_values(
    by='importance',
    ascending=False
)

# select top N features (top 150)
top_features_index = top.head(150).index
X_train_selected = X_train_tfidf[:, top_features_index]
X_test_selected = X_test_tfidf[:, top_features_index]

In [ ]:
# train the RandomForest model with selected features & experimented with parameters
rf_feat = RandomForestClassifier(n_estimators=150, max_depth=12, random_state=SEED)
rf_feat.fit(X_train_selected, y_train_tfidf)

# evaluate the model
y_pred = rf_feat.predict(X_test_selected)
print(classification_report(y_test_tfidf, y_pred))